# 01 · Preparação dos dados | CRISP-DM

**Objetivo:** medir o efeito das regras pendentes e construir features com rastreabilidade temporal. Desenvolvimento de Lucas Menghi com assistência de agente, revisão e aprovação humana: [governança](docs/governance.md).

Este notebook avança com a definição experimental já documentada: compras positivas brutas, duplicatas e códigos especiais preservados. As variantes de limpeza são comparações, não aprovações de negócio. Não escolhemos uma regra porque ela produz menor erro; regras precisam corresponder ao que queremos medir.

> Registro histórico desta etapa. O estado final, incluindo a avaliação de novembro, está em [03_final_model.ipynb](03_final_model.ipynb).

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "src/retail.py").exists() else Path.cwd() / "Online Retail"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.retail import (locate_data, read_transactions, candidate_sales, development_table,
                        FEATURES, RESPONSE, TEST_REFERENCE, temporal_fold)
pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
(ROOT / "reports").mkdir(exist_ok=True)
(ROOT / "data/processed").mkdir(parents=True, exist_ok=True)

transactions = read_transactions(locate_data(ROOT))
dev = transactions.loc[transactions.InvoiceDate < TEST_REFERENCE].copy()

## 1. Sensibilidade das regras

Comparamos preservação, deduplicação exata, exclusão de uma lista explícita de códigos não usuais e ambas as ações. A lista está em `src/retail.py`; não é um detector universal de produtos. Os relatórios mostram linhas, clientes, valor positivo e efeito na variável resposta. As populações podem mudar entre variantes; diferenças não são efeito causal.

Duplicatas exatas não comprovam erro de ingestão. Manter a regra inicial evita uma alteração silenciosa enquanto mensuramos sua consequência.

In [2]:
variants = {"initial": {}, "deduplicated": {"deduplicate": True},
            "merchandise_candidate": {"merchandise_only": True},
            "both": {"deduplicate": True, "merchandise_only": True}}
rows = []
for name, options in variants.items():
    s = candidate_sales(dev, **options)
    sample = development_table(transactions, **options)
    rows.append({"variant": name, "rows": len(s), "customers": s.CustomerID.nunique(),
                 "positive_value_gbp": s.value_gbp.sum(), "customer_references": len(sample),
                 "zero_response_pct": sample[RESPONSE].eq(0).mean() * 100,
                 "response_sum_exposures_gbp": sample[RESPONSE].sum()})
sensitivity = pd.DataFrame(rows)
sensitivity["value_change_pct"] = (sensitivity.positive_value_gbp / sensitivity.positive_value_gbp.iloc[0] - 1) * 100
sensitivity.to_csv(ROOT / "reports/rule_sensitivity.csv", index=False)
display(sensitivity)

,variant,rows,customers,positive_value_gbp,customer_references,zero_response_pct,response_sum_exposures_gbp,value_change_pct
0,initial,316049,3974,"7,231,397.73",15376,67.16,"4,003,001.05",0.00
1,deduplicated,312498,3974,"7,213,812.84",15376,67.16,"3,995,774.85",-0.24
2,merchandise_candidate,314797,3970,"7,105,690.45",15349,67.22,"3,929,746.78",-1.74
3,both,311251,3970,"7,088,465.56",15349,67.22,"3,922,520.58",-1.98


## 2. Revisão de extremos e cancelamentos

Um valor alto pode ser um pedido real, ajuste ou lançamento revertido. Procuramos, somente no desenvolvimento, coincidências de cliente, produto, quantidade absoluta e preço entre vendas e cancelamentos. A comparação é um indício: não há chave que vincule a devolução à linha original e não fazemos reconciliação automática.

Mesmo que um cancelamento posterior seja identificado, usá-lo para apagar uma compra de uma feature anterior ao cancelamento introduziria informação futura. Esta versão mantém a resposta bruta.

In [3]:
sales = candidate_sales(dev)
cancel = dev.loc[dev.InvoiceNo.str.upper().str.startswith("C", na=False) & dev.CustomerID.notna()].copy()
cancel["abs_qty"] = cancel.Quantity.abs()
extreme = sales.nlargest(10, "value_gbp").copy()
evidence = []
for _, row in extreme.iterrows():
    match = cancel.loc[cancel.CustomerID.eq(row.CustomerID) & cancel.StockCode.eq(row.StockCode)
                       & cancel.abs_qty.eq(row.Quantity) & cancel.UnitPrice.eq(row.UnitPrice)
                       & cancel.InvoiceDate.ge(row.InvoiceDate)]
    evidence.append({"product": row.StockCode, "description": row.Description,
                     "sale_date": row.InvoiceDate, "gross_value_gbp": row.value_gbp,
                     "possible_later_cancellations": len(match),
                     "first_possible_cancellation": match.InvoiceDate.min()})
extreme_review = pd.DataFrame(evidence)
extreme_review.to_csv(ROOT / "reports/extreme_review.csv", index=False)
display(extreme_review)

,product,description,sale_date,gross_value_gbp,possible_later_cancellations,first_possible_cancellation
0,23166,MEDIUM CERAMIC TOP STORAGE JAR,2011-01-18 10:01:00,"77,183.60",1,2011-01-18 10:17:00
1,22502,PICNIC BASKET WICKER 60 PIECES,2011-06-10 15:28:00,"38,970.00",0,NaT
2,POST,POSTAGE,2011-05-03 13:46:00,"8,142.75",0,NaT
3,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,2011-09-20 11:05:00,"7,144.72",0,NaT
4,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,2011-01-11 12:55:00,"6,539.40",1,2011-04-18 13:08:00
5,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,2011-04-18 13:20:00,"6,539.40",0,NaT
6,23084,RABBIT NIGHT LIGHT,2011-10-27 12:11:00,"4,992.00",0,NaT
7,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2011-01-11 12:55:00,"4,921.50",1,2011-04-18 13:08:00
8,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2011-04-18 13:20:00,"4,632.00",0,NaT
9,48185,DOORMAT FAIRY CAKE,2011-01-11 12:57:00,"4,522.50",1,2011-04-18 13:08:00


## 3. Features e variável resposta

- Recência: dias desde a última compra anterior à referência.
- Alcance observado: dias desde a primeira compra dentro da janela de 90 dias; não equivale ao tempo de relacionamento.
- Gasto, pedidos distintos, produtos distintos, unidades e ticket em 30/60/90 dias.
- Tendência: gasto nos últimos 30 dias menos gasto nos 30 dias anteriores.

Todas as features usam eventos anteriores à referência. Zero em uma janela sem compras é estrutural. `CustomerID` identifica linhas, mas não entra no estimador. País e variáveis de calendário ficam para experimentos posteriores; o histórico curto dificulta estimar sazonalidade.

A fonte não contém horário de disponibilidade. Assumimos que o evento já estava disponível; não é possível provar essa hipótese com este arquivo.

In [4]:
table = development_table(transactions)
assert table.reference_date.max() < TEST_REFERENCE
assert table.response_end.max() <= TEST_REFERENCE
table.to_csv(ROOT / "data/processed/development.csv", index=False)
display(table[FEATURES + [RESPONSE]].describe().T)
print("Cliente/referência:", len(table), "| Features:", len(FEATURES))
fold_rows = []
for t in sorted(table.reference_date.unique())[3:]:
    train, validation = temporal_fold(table, t)
    fold_rows.append({"validation_reference": t, "train_rows": len(train),
                      "last_train_response_end": train.response_end.max(), "validation_rows": len(validation)})
folds = pd.DataFrame(fold_rows)
folds.to_csv(ROOT / "reports/temporal_folds.csv", index=False)
display(folds)

,count,mean,std,min,25%,50%,75%,max
recency_days,"15,376.00",35.43,25.76,0.16,13.28,29.51,54.53,89.67
observed_span_days,"15,376.00",53.22,26.40,0.16,31.38,56.56,77.30,89.67
spend_30d,"15,376.00",326.01,"1,380.55",0.00,0.00,20.80,343.04,"75,412.64"
orders_30d,"15,376.00",0.70,1.04,0.00,0.00,1.00,1.00,28.00
products_30d,"15,376.00",13.20,25.21,0.00,0.00,1.00,19.00,639.00
units_30d,"15,376.00",191.06,726.11,0.00,0.00,3.00,198.00,"29,300.00"
ticket_30d,"15,376.00",214.92,525.97,0.00,0.00,18.85,307.50,"21,880.44"
spend_60d,"15,376.00",626.48,"2,247.13",0.00,101.03,302.92,596.83,"91,532.10"
orders_60d,"15,376.00",1.34,1.69,0.00,1.00,1.00,2.00,40.00
products_60d,"15,376.00",23.92,34.30,0.00,3.00,16.00,32.00,763.00


Cliente/referência: 15376 | Features: 18


,validation_reference,train_rows,last_train_response_end,validation_rows
0,2011-06-01,5349,2011-05-31,1959
1,2011-07-01,7308,2011-07-01,1969
2,2011-08-01,9277,2011-07-31,2025
3,2011-09-01,11302,2011-08-31,1923
4,2011-10-01,13225,2011-10-01,2151


## 4. Decisão para o experimento inicial

Prosseguimos com regras iniciais preservadas para estabelecer referências de desempenho. Nenhuma variante é promovida como regra final. Reveja os impactos acima e [o registro de decisões](docs/decisions.md).

Os testes automatizados verificam bordas de janela, manutenção dos zeros, exclusão de clientes conhecidos apenas no futuro, maturação da resposta e invariância de features quando os eventos futuros são alterados. Esses testes não comprovam as hipóteses de negócio.

**Próximo notebook:** comparar regras históricas, Ridge, Tweedie e boosting sob os mesmos folds, sem abrir novembro.